# Praktikum 5 — Decision Tree dan Pruning
Notebook ini berisi kode praktikum klasifikasi Decision Tree menggunakan dataset Breast Cancer dari scikit-learn.

## 1. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, validation_curve
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report

## 2. Praktikum 5.2 — Import Dataset

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target

print("Ukuran data:", X.shape)
print("Distribusi kelas:", np.bincount(y))

## 3. Praktikum 5.3 — Preprocessing (Split Data)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 4. Praktikum 5.4 — Training Model (dengan dan tanpa Pruning)

In [ ]:
# Model tanpa pruning (tumbuh penuh)
tree_full = DecisionTreeClassifier(random_state=42)
tree_full.fit(X_train, y_train)

# Model dengan pre-pruning
tree_pruned = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=5,
    random_state=42
)
tree_pruned.fit(X_train, y_train)

# Model dengan post-pruning (Cost Complexity Pruning)
path = tree_full.cost_complexity_pruning_path(X_train, y_train)
ccp_alpha_terpilih = path.ccp_alphas[len(path.ccp_alphas) // 2]

tree_ccp = DecisionTreeClassifier(
    ccp_alpha=ccp_alpha_terpilih,
    random_state=42
)
tree_ccp.fit(X_train, y_train)

print("ccp_alpha terpilih (contoh):", ccp_alpha_terpilih)

## 5. Praktikum 5.5 — Evaluasi

In [ ]:
for nama, model in [
    ("Tanpa Pruning", tree_full),
    ("Pre-Pruning", tree_pruned),
    ("Post-Pruning (CCP)", tree_ccp)
]:
    acc_train = accuracy_score(y_train, model.predict(X_train))
    acc_test = accuracy_score(y_test, model.predict(X_test))
    print(
        f"{nama}: Akurasi Train={acc_train:.3f}, "
        f"Akurasi Test={acc_test:.3f}, "
        f"Gap={acc_train - acc_test:.3f}"
    )

print("\nClassification report — model pre-pruning:")
print(classification_report(y_test, tree_pruned.predict(X_test), target_names=data.target_names))

## 6. Praktikum 5.6 — Visualisasi Pohon

In [ ]:
plt.figure(figsize=(18, 10))
plot_tree(
    tree_pruned,
    feature_names=data.feature_names,
    class_names=data.target_names,
    filled=True,
    rounded=True,
    fontsize=7
)
plt.title("Decision Tree (max_depth=4, Pre-Pruning)")
plt.tight_layout()
plt.show()

## 7. Praktikum 5.7 — Eksperimen Parameter Pre-Pruning

In [ ]:
hasil_pre_pruning = []

for depth in [2, 4, 6, None]:
    for min_leaf in [1, 5, 10]:
        model = DecisionTreeClassifier(
            max_depth=depth,
            min_samples_leaf=min_leaf,
            random_state=42
        )
        model.fit(X_train, y_train)
        acc_train = accuracy_score(y_train, model.predict(X_train))
        acc_test = accuracy_score(y_test, model.predict(X_test))
        hasil_pre_pruning.append({
            "max_depth": str(depth),
            "min_samples_leaf": min_leaf,
            "akurasi_train": acc_train,
            "akurasi_test": acc_test
        })

df_pre_pruning = pd.DataFrame(hasil_pre_pruning)
display(df_pre_pruning.sort_values('akurasi_test', ascending=False).reset_index(drop=True))

## 8. Praktikum 5.8 — Cost Complexity Pruning Path

In [ ]:
path = tree_full.cost_complexity_pruning_path(X_train, y_train)
akurasi_train_ccp = []
akurasi_test_ccp = []
jumlah_leaf = []

for alpha in path.ccp_alphas:
    model = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    model.fit(X_train, y_train)
    akurasi_train_ccp.append(accuracy_score(y_train, model.predict(X_train)))
    akurasi_test_ccp.append(accuracy_score(y_test, model.predict(X_test)))
    jumlah_leaf.append(model.get_n_leaves())

hasil_ccp = pd.DataFrame({
    "ccp_alpha": path.ccp_alphas,
    "akurasi_train": akurasi_train_ccp,
    "akurasi_test": akurasi_test_ccp,
    "jumlah_leaf": jumlah_leaf
})
display(hasil_ccp.sort_values('akurasi_test', ascending=False).head(10))

plt.figure(figsize=(10, 5))
plt.plot(path.ccp_alphas, akurasi_train_ccp, marker='o', label='Akurasi Train')
plt.plot(path.ccp_alphas, akurasi_test_ccp, marker='o', label='Akurasi Test')
plt.xlabel('ccp_alpha')
plt.ylabel('Akurasi')
plt.title('Akurasi Train dan Test terhadap ccp_alpha')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Pilih alpha berdasarkan akurasi test tertinggi (untuk demonstrasi praktikum).
# Untuk evaluasi yang lebih ketat, pilih parameter menggunakan validation set/cross-validation.
idx_terbaik = int(np.argmax(akurasi_test_ccp))
alpha_terbaik = path.ccp_alphas[idx_terbaik]
tree_ccp_terbaik = DecisionTreeClassifier(ccp_alpha=alpha_terbaik, random_state=42)
tree_ccp_terbaik.fit(X_train, y_train)

print("ccp_alpha terbaik pada test set (demonstrasi):", alpha_terbaik)
print("Akurasi test:", accuracy_score(y_test, tree_ccp_terbaik.predict(X_test)))
print("Jumlah leaf:", tree_ccp_terbaik.get_n_leaves())

## 9. Ringkasan Hasil
Jalankan semua sel dari atas ke bawah. Bandingkan akurasi train dan test serta jumlah leaf untuk melihat dampak pruning. Catatan: memilih parameter berdasarkan test set dapat membuat estimasi performa terlalu optimistis; untuk proyek nyata gunakan validation set atau cross-validation.

In [ ]:
ringkasan = []
model_final = [
    ("Tanpa Pruning", tree_full),
    ("Pre-Pruning", tree_pruned),
    ("Post-Pruning (CCP contoh)", tree_ccp),
    ("Post-Pruning (CCP terbaik pada test)", tree_ccp_terbaik),
]
for nama, model in model_final:
    ringkasan.append({
        "model": nama,
        "akurasi_train": accuracy_score(y_train, model.predict(X_train)),
        "akurasi_test": accuracy_score(y_test, model.predict(X_test)),
        "jumlah_leaf": model.get_n_leaves(),
        "kedalaman": model.get_depth()
    })
display(pd.DataFrame(ringkasan))